# Parameter inference with HNN

The **Human Neocortical Neurosolver (HNN)** simulates cortical circuits containing pyramidal and basket cells in layers 2/3 and 5. It predicts current dipoles in nAm, comparable to source-localized MEG/EEG signals. Here we use a tiny cortical column to learn a posterior over two external-input parameters with BayesFlow. See [HNN-core](https://joss.theoj.org/papers/10.21105/joss.05848) and the [batch simulation tutorial](https://jonescompneurolab.github.io/textbook/content/08_using_hnn_api/batch_simulation.html).

## What do we estimate?

| HNN parameter | Meaning | Prior |
| --- | --- | --- |
| `mu` | Mean arrival time of a proximal evoked input, in ms. | $$\mu \sim \mathcal{U}(25, 45)$$ |
| `weight_pyr` | AMPA input weight shared by L2/3 and L5 pyramidal cells, in µS. | $$\log w_{\mathrm{pyr}} \sim \mathcal{U}(\log 10^{-4}, \log(8\times10^{-4}))$$ |

**Observables:** the aggregate current-dipole waveform at 66 time points, from 10 to 75 ms. Timing should shift the response; weight should change its amplitude. We smooth for 5 ms, subtract the pre-input baseline, and add Gaussian measurement noise with SD $2\times10^{-6}\;\mathrm{nAm}$.

We fix the 3×3 pyramidal mesh, local connectivity, basket-cell input weight ($10^{-4}\;\mu\mathrm{S}$), input-time spread (3 ms), and synaptic delays. Each simulation uses fresh input-event and measurement-noise randomness. This is a small identifiability demo, rather than a fit of all HNN parameters.

## Setup

Run `uv sync` and select this repo's Python kernel. The included offline dataset lets you train immediately without installing NEURON. To regenerate it, use Linux, macOS, or WSL with a C/C++ compiler and run `uv sync --extra hnn`. HNN 0.6.1 builds its mechanisms using NEURON 8.2.7; see [HNN installation](https://jonescompneurolab.github.io/textbook/content/01_getting_started/installation.html).

In [ ]:
import bayesflow as bf
import keras

from tutorials.helpers import hnn

In [ ]:
labels = hnn.PARAMETER_LABELS

## Simulate once, train offline

HNN's `BatchSimulate` runs independent parameter draws on fresh network copies. The simulation and waveform utilities live in [hnn.py](helpers/hnn.py). Load the stored simulations below; set `regenerate=True` to generate them again with HNN installed.

In [ ]:
train, validation, test = hnn.load_dataset(regenerate=False)
print({key: values.shape for key, values in train.items()})

Each panel overlays the same 40 simulated dipole waveforms. The horizontal axis is time in ms and the vertical axis is current dipole in nAm. Color indicates input arrival time on the left and pyramidal AMPA weight in nS on the right, showing how these parameters affect response timing and amplitude.

In [ ]:
fig = hnn.plot_waveforms(train)

## Offline NPE through an adapter

The adapter log-transforms the positive weight, concatenates both parameters, and passes the dipole vector as the condition. BayesFlow reverses the transform when sampling, returning weights in µS. A small coupling flow learns $$q_\phi(\mu, w_{\mathrm{pyr}} \mid \mathrm{dipole})$$ from the fixed training set; validation monitors generalization. No simulator is called during training.

In [ ]:
### Your code here
adapter = None

### Your code here
workflow = None

### Your code here
history = None

The loss plot compares training and validation loss across epochs. Validation uses separate simulations and helps reveal overfitting.

In [ ]:
### Your code here
fig = None

## Does it recover held-out parameters?

Next, we condition on 128 unseen dipole waveforms and perform diagnostic checks that become much cheaper with amortized inference.

In [ ]:
### Your code here
posterior_samples = None

The recovery plot compares posterior medians and 95% credible intervals with the true simulation parameters. Estimates should follow the diagonal; $r$ reports their correlation with ground truth. Both axes use the same units.

In [ ]:
### Your code here
fig = None

The calibration plot shows the rank ECDF minus the uniform ECDF across held-out simulations. Curves near zero indicate calibration; gray bands show the 95% simultaneous bounds expected under uniform ranks.

In [ ]:
### Your code here
fig = None

Summarize recovery, calibration, and posterior contraction in a table:

In [ ]:
### Your code here
table = None

## Next steps

Write down the next steps of the workflow and potential directions for improvement.